# Admission Mortality - Inpatient and ICU

In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics, preprocess_inference_results
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

/home/henri/ethos-benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.mimic_icu import ICUMortalityDataset
from src.ethos.datasets.hospital_mortality import HospitalMortalityDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = ICUMortalityDataset(path)
sum_adm = sum([True if d[1]['expected'] == 'MEDS_DEATH' else False for d in dataset])
sum_adm#/len(dataset)

10997 admissions, 10997 discharges, 4364 deaths found.
Found 10997 hospital stays with an outcome token.


775

In [3]:
import json
true_ids = [d[1]['hadm_id'] for d in dataset if d[1]['expected'] == 'MEDS_DEATH']
with open("true_ids.json", "w") as f:
    json.dump(true_ids, f)

In [4]:
for i in range(len(dataset)): 
    if dataset[i][1]['hadm_id'] == 20345487.0:
        print(i, dataset[i][1])
        break

5 {'expected': 'HOSPITAL_DISCHARGE', 'true_token_dist': 299, 'true_token_time': 82521000000, 'patient_id': 10002155, 'prediction_time': 5086523640000000, 'data_idx': 3241, 'hadm_id': 20345487, 'icu_stay_id': None}


In [5]:
import pandas as pd
df_token = pd.read_parquet(f"{path}/13_inject_time_intervals/0.parquet")

In [6]:
df_id = df_token[df_token['subject_id'].isin([10002155])]
df_id[df_id.code.str.contains('HOSPITAL') | df_id.code.str.contains('ICU_') | df_id.code.str.contains('DEATH')]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
153,10002155,2128-07-29 14:50:00,HOSPITAL_ADMISSION,NaN,EW_EMER,28439444.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
342,10002155,2128-07-31 18:00:00,HOSPITAL_DISCHARGE,NaN,HOME,28439444.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
385,10002155,2129-08-04 11:00:00,HOSPITAL_ADMISSION,NaN,EW_EMER,23822395.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
460,10002155,2129-08-04 12:45:00,ICU_ADMISSION,NaN,ICU_TYPE//Coronary Care Unit (CCU),23822395.0,NaN,33685454.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
461,10002155,2129-08-04 12:45:00,ICU_TYPE//CORONARY_CARE_UNIT_(CCU),NaN,ICU_TYPE//Coronary Care Unit (CCU),23822395.0,NaN,33685454.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
871,10002155,2129-08-10 17:02:38,ICU_DISCHARGE,NaN,ICU_TYPE//Coronary Care Unit (CCU),23822395.0,NaN,33685454.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1280,10002155,2129-08-18 16:53:00,HOSPITAL_DISCHARGE,NaN,CHRONIC/LONG TERM ACUTE CARE,23822395.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1886,10002155,2129-12-23 21:59:00,HOSPITAL_ADMISSION,NaN,EW_EMER,28976727.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN
2054,10002155,2129-12-25 19:18:00,HOSPITAL_DISCHARGE,NaN,HOME HEALTH CARE,28976727.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2614,10002155,2130-09-23 19:59:00,HOSPITAL_ADMISSION,NaN,EW_EMER,28994087.0,NaN,NaN,Other,ENGLISH,MARRIED,WHITE,NaN,NaN,NaN,NaN,NaN


In [12]:
df_11204531 = df_token[df_token['subject_id'].isin([11204531.0])]
df_11204531[df_11204531.code.str.contains('DEATH')]

,subject_id,time,code,numeric_value,text_value,hadm_id,edstay_id,icustay_id,insurance,language,marital_status,race,drg_severity,drg_mortality,emar_id,emar_seq,priority
2117118,11204531,2130-09-20 04:52:00,MEDS_DEATH,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# MIMIC

In [ ]:
results_dir = PROJECT_ROOT / "results"
hosp_mort_result_dir = results_dir / Task.HOSPITAL_MORTALITY
icu_mort_result_dir = results_dir / Task.ICU_MORTALITY


def preprocess_hosp_mortality_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual") == ST.DEATH,
        expected_expr=pl.col("expected") == ST.DEATH,
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )
    
def preprocess_icu_mortality_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual") == ST.DEATH,
        expected_expr=pl.col("expected") == ST.DEATH,
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

## Hospital Mortality

In [ ]:
hospital_mortality_results = []
for result_fp in hosp_mort_result_dir.iterdir():
    df = preprocess_hosp_mortality_results(result_fp)
    rep_num = df["counts"].mean()
    plt.figure(figsize=(10, 5))
    plt.suptitle(f"Hospital Mortality\n{result_fp.name} (rep_num={rep_num:.2f})")
    plt.subplot(1, 2, 1)
    res_full = compute_and_print_metrics(df["expected"], df["actual"], "Full")

    plt.subplot(1, 2, 2)
    # hf_df = df.filter(pl.col("patient_id").is_in(heart_failure_cohort))
    # res_hf = compute_and_print_metrics(hf_df["expected"], hf_df["actual"], "HF patients")

    hospital_mortality_results.append(
        {
            "name": result_fp.name,
            "auc": res_full["fitted_auc"],
            # "auc_hf": res_hf["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.tight_layout()
    plt.show()
hospital_mortality_results = pl.from_dicts(hospital_mortality_results).sort("auc", descending=True)

In [ ]:
hospital_mortality_results

## ICU Mortality

In [ ]:
icu_mortality_results = []
for result_fp in icu_mort_result_dir.iterdir():
    df = preprocess_icu_mortality_results(result_fp)
    rep_num = df["counts"].mean()
    plt.figure(figsize=(10, 5))
    plt.suptitle(f"ICU Mortality\n{result_fp.name} (rep_num={rep_num:.2f})")
    plt.subplot(1, 2, 1)
    res_full = compute_and_print_metrics(df["expected"], df["actual"], "Full")

    plt.subplot(1, 2, 2)
    # hf_df = df.filter(pl.col("patient_id").is_in(heart_failure_cohort))
    # res_hf = compute_and_print_metrics(hf_df["expected"], hf_df["actual"], "HF patients")

    icu_mortality_results.append(
        {
            "name": result_fp.name,
            "auc": res_full["fitted_auc"],
            # "auc_hf": res_hf["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.tight_layout()
    plt.show()
icu_mortality_results = pl.from_dicts(icu_mortality_results).sort("auc", descending=True)

In [ ]:
icu_mortality_results